In [8]:
%pwd

'c:\\Users\\Amal\\Desktop\\End to End project Mlops mlflow github\\End-to-End-project-Mlops-Mlflow\\research'

In [9]:
%cd ..

c:\Users\Amal\Desktop\End to End project Mlops mlflow github\End-to-End-project-Mlops-Mlflow


C:\Users\Amal\AppData\Roaming\Python\Python39\site-packages\IPython\core\magics\osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [10]:
%pwd

'c:\\Users\\Amal\\Desktop\\End to End project Mlops mlflow github\\End-to-End-project-Mlops-Mlflow'

In [71]:
import pandas as pd

In [72]:
data = pd.read_csv("artifacts/data_ingestion/Housing.csv")

In [73]:
data

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,yes,no,yes,no,no,2,no,unfurnished
541,1767150,2400,3,1,1,no,no,no,no,no,0,no,semi-furnished
542,1750000,3620,2,1,1,yes,no,no,no,no,0,no,unfurnished
543,1750000,2910,3,1,1,no,no,no,no,no,0,no,furnished


In [74]:
set(data["furnishingstatus"])

{'furnished', 'semi-furnished', 'unfurnished'}

Convert the input data from strings to integer values

In [ ]:
from sklearn.preprocessing import LabelEncoder

column_name = ["mainroad", "guestroom", "basement", "hotwaterheating", "airconditioning", "prefarea", "furnishingstatus"]

In [ ]:
encoders = {}

for col in column_name:
    encoder = LabelEncoder()
    data[col] = encoder.fit_transform(data[col]) 
    encoders[col] = encoder
    print(col, set(data[col]))
    
    # show the mapping of original values to encoded numbers
    print(f" {dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))}")
    print()

mainroad {0, 1}
 {'no': 0, 'yes': 1}

guestroom {0, 1}
 {'no': 0, 'yes': 1}

basement {0, 1}
 {'no': 0, 'yes': 1}

hotwaterheating {0, 1}
 {'no': 0, 'yes': 1}

airconditioning {0, 1}
 {'no': 0, 'yes': 1}

prefarea {0, 1}
 {'no': 0, 'yes': 1}

furnishingstatus {0, 1, 2}
 {'furnished': 0, 'semi-furnished': 1, 'unfurnished': 2}



In [95]:
data

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,1,0,0,0,1,2,1,0
1,12250000,8960,4,4,4,1,0,0,0,1,3,0,0
2,12250000,9960,3,2,2,1,0,1,0,0,2,1,1
3,12215000,7500,4,2,2,1,0,1,0,1,3,1,0
4,11410000,7420,4,1,2,1,1,1,0,1,2,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,1,0,1,0,0,2,0,2
541,1767150,2400,3,1,1,0,0,0,0,0,0,0,1
542,1750000,3620,2,1,1,1,0,0,0,0,0,0,2
543,1750000,2910,3,1,1,0,0,0,0,0,0,0,0


Update the entity

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    transf_data_file: str
    encoder_file: str
    all_schema: dict # read the file schema.yaml and save it in a dictionary format
    column_str_name: str

Update the configuration manager

In [3]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

In [ ]:

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root]) 

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        schema = self.schema.COLUMNS

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir = config.root_dir,
            data_path = config.data_path,
            transf_data_file = config.transf_data_file,
            encoder_file = config.encoder_file,
            all_schema = self.schema,
            column_str_name = [k for k, v in schema.items() if v == "str"]
        )

        return data_transformation_config

Update the components

In [5]:
import joblib
import os
from mlProject import logger
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.preprocessing import LabelEncoder

In [ ]:

class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        
        
    def train_test_splitting(self):
        
        data = pd.read_csv(self.config.data_path)
  
       # split the data into training and test sets : (0.80, 0.20)
        train_data, test_data= train_test_split(data, test_size=0.20, random_state=42)
        
        logger.info("Split data into training and test sets")
        logger.info("train shape: " + str(train_data.shape))  
        logger.info("test shape: "+ str(test_data.shape))
        
        return train_data, test_data
    
    ## Note: You can add other data transformation techniques such as PCA 
    # You can perform all kinds of exploratory data analysis (EDA) here before passing the data to the model

    def Normalize_data (self, train_data, test_data):
        
        # column_name = ["mainroad", "guestroom", "basement", "hotwaterheating","airconditioning", "prefarea", "furnishingstatus"]
        column_name = self.config.column_str_name
        
        # 1. Train data
        input_train_data = train_data.drop(self.config.all_schema.TARGET_COLUMN.name, axis=1)
        
        # convert the training input data from strings to integer values
        encoders = {}

        for col in column_name:
            encoder = LabelEncoder()
            input_train_data[col] = encoder.fit_transform(input_train_data[col])  
            encoders[col] = encoder
         
         # save encoders
        joblib.dump(encoders, self.config.encoder_file)

         # get mean, max and min values of the training input data for normalizing new test data
        mean_input_data, max_input_data, min_input_data = input_train_data.mean(), input_train_data.max(), input_train_data.min()
        
        # save mean, max and min values of the training input data for normalizing new test data
        transf_data = pd.DataFrame({
                "mean_input_data": mean_input_data,
                "max_input_data": max_input_data,
                "min_input_data": min_input_data
                })

        transf_data.to_csv(self.config.transf_data_file)


        # normalize the training input data using mean normalization.
        input_normalized_train_data = (input_train_data - mean_input_data)/(max_input_data-min_input_data)

        # concatenate the output data with the normalized input data and save it
        data_train_normalized = pd.concat([input_normalized_train_data , train_data[self.config.all_schema.TARGET_COLUMN.name]], axis=1)
        data_train_normalized.to_csv(os.path.join(self.config.root_dir, "train.csv"),index = False)
        
        # 2. Test data
        input_test_data = test_data.drop(self.config.all_schema.TARGET_COLUMN.name, axis=1)
        
        # convert the test input data from strings to integer values
        for col in column_name:
            input_test_data[col] = encoders[col].transform(input_test_data[col]) 
        
        # normalize the test input data using mean normalization
        input_normalized_test_data = (input_test_data - mean_input_data)/(max_input_data-min_input_data)

        # concatenate the output data with the normalized test input data and save it
        data_test_normalized = pd.concat([input_normalized_test_data , test_data[self.config.all_schema.TARGET_COLUMN.name]], axis=1)
        data_test_normalized.to_csv(os.path.join(self.config.root_dir, "test.csv"),index = False)

Update the pipline

In [22]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    train_data, test_data = data_transformation.train_test_splitting()
    data_transformation. Normalize_data(train_data, test_data)
except Exception as e:
    raise e

[2026-06-25 16:35:23,880: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-25 16:35:23,887: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-25 16:35:23,899: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-06-25 16:35:23,902: INFO: common: created directory at: artifacts]


[2026-06-25 16:35:23,906: INFO: common: created directory at: artifacts/data_transformation]
[2026-06-25 16:35:23,926: INFO: 3046005871: Split data into training and test sets]
[2026-06-25 16:35:23,930: INFO: 3046005871: train shape: (436, 13)]
[2026-06-25 16:35:23,933: INFO: 3046005871: test shape: (109, 13)]
